# Data Science Programming Project
## Analysing Ethnicity, Stop & Search, Prosecutions and Convictions in the UK
### Open Government Datasets — data.gov.uk

**Datasets used:**
- Population by Ethnicity and Region (2021)
- Prosecutions and Convictions (2009–2017)
- Stop and Search (2006–2023)

**Objective:** Explore relationships between ethnic population distribution, stop and search activity, and criminal justice outcomes (prosecutions and convictions) across different regions, time periods, age groups, and offence types in England and Wales.

## 1. Importing Libraries

In [5]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Global plot styling
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.family': 'Arial', 'axes.titlesize': 20, 'axes.labelsize': 16,
                     'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 14})
pastel = sns.color_palette('pastel')
fmt_comma = mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')

## 2. Data Loading and Preparation

### 2.1 Loading the Datasets

Three datasets are loaded using `pd.read_csv`. Latin-1 encoding is used to prevent invalid byte errors common in government data exports.

In [9]:
import os
from pathlib import Path

# Resolve path relative to the notebook file itself
os.chdir(Path(__file__).resolve().parent.parent if '__file__' in dir() else Path().resolve())

df1 = pd.read_csv("Data/population-by-ethnicity-and-region-2021.csv", encoding="latin-1")
df2 = pd.read_csv("Data/prosecutions-and-convictions.csv",             encoding="latin-1")
df3 = pd.read_csv("Data/stop-and-search-2006-2023.csv",                encoding="latin-1")

# Preserve originals
d1f = df1.copy()
d2f = df2.copy()
d3f = df3.copy()

print(f"Population dataset:        {df1.shape[0]:,} rows, {df1.shape[1]} columns")
print(f"Prosecutions dataset:      {df2.shape[0]:,} rows, {df2.shape[1]} columns")
print(f"Stop & Search dataset:     {df3.shape[0]:,} rows, {df3.shape[1]} columns")

FileNotFoundError: [Errno 2] No such file or directory: 'Data/population-by-ethnicity-and-region-2021.csv'

### 2.2 Initial Data Exploration

In [ ]:
for name, df in [('Population', df1), ('Prosecutions', df2), ('Stop & Search', df3)]:
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    df.info()

### 2.3 Null Value Analysis and Cleaning

**df1 (Population):** No null values after dropping columns below the 50% threshold.

**df2 (Prosecutions):** Rows where prosecutions (the denominator) are null are dropped entirely, as they cannot contribute to analysis.

**df3 (Stop & Search):** Numerical measure columns (counts, rates, proportions) are filled with 0. Population by ethnicity is imputed using the column median.

In [ ]:
# df1
df1.dropna(axis=1, thresh=len(df1)//2, inplace=True)

# df2
df2.dropna(axis=1, thresh=len(df2)//2, inplace=True)
df2.dropna(axis=0, inplace=True)

# df3
df3.dropna(axis=1, thresh=len(df3)//2, inplace=True)
fill_zero_cols = [
    'proportion_of_total_stop_and_searches_of_this_ethnicity_in_the_financial_year_excludes_unreported',
    'rate_per_1_000_population_by_ethnicity',
    'total_number_of_stop_and_search_carried_out_in_this_year_in_this_area_excluding_cases_where_the_ethnicity_was_unreported',
    'number_of_stop_and_searches'
]
df3[fill_zero_cols] = df3[fill_zero_cols].fillna(0)
df3['population_by_ethnicity'] = df3['population_by_ethnicity'].fillna(df3['population_by_ethnicity'].median())

# Time column for df3
df3['year'] = df3['time'].str.split('/').str[0].astype(int)

print("Cleaning complete. Remaining nulls:")
for name, df in [('df1', df1), ('df2', df2), ('df3', df3)]:
    print(f"  {name}: {df.isnull().sum().sum()} null values")

## 3. Ethnic Population Distribution (df1)

### 3.1 National Ethnic Population Breakdown

We first establish the demographic baseline using the 2021 Census population data. This is critical context for all downstream analyses — raw counts of prosecutions or stop and searches must always be interpreted against population proportions to identify genuine disproportionality.

In [ ]:
# Broad ethnicity groups (ONS 5+1 classification) at national level
df1_broad = df1[df1['Ethnicity_type'] == 'ONS 2021 5+1']
df1_national = df1_broad[df1_broad['Geography'] == 'All - England And Wales'].copy()
df1_national = df1_national[df1_national['Ethnicity'] != 'All'].sort_values('Ethnic Population', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Bar chart
bars = ax1.bar(df1_national['Ethnicity'], df1_national['Ethnic Population'],
               color=pastel, edgecolor='black')
ax1.yaxis.set_major_formatter(fmt_comma)
ax1.set_title('Ethnic Population — England & Wales 2021', pad=15)
ax1.set_xlabel('Ethnicity')
ax1.set_ylabel('Population')
ax1.tick_params(axis='x', rotation=30)
for b in bars:
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 50000,
             f"{b.get_height()/1e6:.1f}M", ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart
ax2.pie(df1_national['Ethnic Population'], labels=df1_national['Ethnicity'],
        autopct='%1.1f%%', startangle=90, colors=pastel, explode=[0.05]*len(df1_national))
ax2.set_title('Population Share by Ethnicity', pad=15)

plt.tight_layout()
plt.show()

print("\nEthnic population proportions (England & Wales 2021):")
df1_national[['Ethnicity','Ethnic Population','Value1']].rename(columns={'Value1':'% of total'}).to_string(index=False)
print(df1_national[['Ethnicity','Ethnic Population','Value1']].rename(columns={'Value1':'% of total'}).to_string(index=False))

**Key observations:**
- White groups constitute **81.7%** of the total population, making them the demographic majority.
- Asian groups are the second largest at **9.3%**, followed by Black at **4.0%**, Mixed at **2.9%**, and Other at **2.1%**.
- This population baseline is essential: a group comprising 81.7% of the population but 50% of prosecutions would be *underrepresented* in criminal justice — while a group comprising 4% of the population but 10% of prosecutions would be *overrepresented*.

### 3.2 Ethnic Population by Region

In [ ]:
df1_regional = df1[df1['Ethnicity_type'] == 'ONS 2021 5+1'].copy()
df1_regional = df1_regional[(df1_regional['Geography'] != 'All - England And Wales') &
                             (df1_regional['Ethnicity'] != 'All')]

pivot = df1_regional.pivot_table(index='Geography', columns='Ethnicity',
                                  values='Ethnic Population', aggfunc='sum')
pivot = pivot.div(pivot.sum(axis=1), axis=0) * 100  # convert to %

pivot.plot(kind='bar', figsize=(18, 8), colormap='Pastel1', edgecolor='black', width=0.75)
plt.title('Ethnic Population Share by Region — England & Wales 2021', pad=15)
plt.xlabel('Region')
plt.ylabel('% of Regional Population')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Ethnicity', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\nRegions with highest non-White population share:")
pivot['Non-White'] = 100 - pivot['White']
print(pivot['Non-White'].sort_values(ascending=False).round(1))

**Key observations:**
- London has by far the highest non-White population share among all regions.
- The North East and Wales have the lowest ethnic diversity.
- This regional variation is important context when interpreting regional differences in prosecutions and stop and search activity — areas with higher non-White populations will naturally show different distributions.

## 4. Prosecutions and Convictions Analysis (df2)

### 4.1 Computing Conviction Rate

The conviction rate is the proportion of prosecutions that result in a conviction. This is a more informative metric than raw counts as it captures the effectiveness of the prosecution.

In [ ]:
df2['conviction_rate'] = (df2['Numerator (Convictions and Magistrates Court and Crown Court)'] /
                          df2['Denominator (Prosecutions at Magistrates Court)'] * 100)
print("Conviction rate column added.")
print(df2[['Ethnicity','Time','conviction_rate']].head(8))

### 4.2 Overall Trend — Prosecutions and Convictions Over Time

In [ ]:
df2_trend = df2[(df2['Sex']=='All') & (df2['Age group']=='All') &
                (df2['Offence group']=='All') & (df2['Police Force Area']=='All') &
                (df2['Ethnicity']!='All')].copy()

df2_time_total = df2_trend.groupby('Time').agg(
    Prosecutions=('Denominator (Prosecutions at Magistrates Court)', 'sum'),
    Convictions=('Numerator (Convictions and Magistrates Court and Crown Court)', 'sum')
).reset_index()
df2_time_total['conviction_rate'] = df2_time_total['Convictions'] / df2_time_total['Prosecutions'] * 100

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.bar(df2_time_total['Time'] - 0.2, df2_time_total['Prosecutions'], width=0.4,
        color='thistle', edgecolor='black', label='Prosecutions')
ax1.bar(df2_time_total['Time'] + 0.2, df2_time_total['Convictions'],  width=0.4,
        color='lightgreen', edgecolor='black', label='Convictions')
ax1.yaxis.set_major_formatter(fmt_comma)
ax1.set_xlabel('Year')
ax1.set_ylabel('Count')
ax1.set_title('Total Prosecutions and Convictions Over Time (All Ethnicities)', pad=15)
ax1.legend()
ax2 = ax1.twinx()
ax2.plot(df2_time_total['Time'], df2_time_total['conviction_rate'],
         color='crimson', marker='o', linewidth=2, label='Conviction Rate %')
ax2.set_ylabel('Conviction Rate (%)', color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')
ax2.set_ylim(0, 110)
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(df2_time_total.to_string(index=False))

**Key observations:**
- Total prosecutions have declined consistently from 2009 to 2017, with a minor uptick in 2009–2010.
- Despite falling prosecution volumes, the conviction rate has risen over the same period — from around **76%** in 2009 to **83%** by 2017.
- **Additional inference:** The rising conviction rate alongside falling prosecution volumes suggests that prosecutorial decision-making has become more selective over time. Cases are less frequently brought to court, but those that are increasingly result in conviction — consistent with CPS efficiency reforms and stricter charging standards introduced during this period.

### 4.3 Conviction Rate by Ethnicity Over Time

In [ ]:
df2_eth_trend = df2_trend.groupby(['Time','Ethnicity']).agg(
    Prosecutions=('Denominator (Prosecutions at Magistrates Court)', 'sum'),
    Convictions=('Numerator (Convictions and Magistrates Court and Crown Court)', 'sum')
).reset_index()
df2_eth_trend['conviction_rate'] = df2_eth_trend['Convictions'] / df2_eth_trend['Prosecutions'] * 100

pivot_cr = df2_eth_trend.pivot(index='Time', columns='Ethnicity', values='conviction_rate')

plt.figure(figsize=(14, 6))
for col in pivot_cr.columns:
    plt.plot(pivot_cr.index, pivot_cr[col], marker='o', linewidth=2, label=col)
plt.title('Conviction Rate by Ethnicity Over Time', pad=15)
plt.xlabel('Year')
plt.ylabel('Conviction Rate (%)')
plt.ylim(65, 95)
plt.legend(title='Ethnicity')
plt.tight_layout()
plt.show()

print("\nMean conviction rate by ethnicity (2009-2017):")
print(pivot_cr.mean().round(1).sort_values(ascending=False))

**Key observations:**
- White and Other inc. Chinese ethnicities consistently have the **highest conviction rates** (~83–85%).
- Asian and Mixed ethnicities have the **lowest conviction rates** (~75–79%).
- All ethnicities show an upward trend in conviction rates over the period.
- **Additional inference:** The persistent gap in conviction rates between ethnic groups may reflect differences in the types of offences prosecuted, legal representation quality, or case selection by the CPS. Asian and Mixed groups seeing lower rates despite being prosecuted could indicate weaker cases are more frequently brought against these groups — or that factors like language barriers affect court outcomes.

### 4.4 Gender Gap in Prosecutions

In [ ]:
df2_gender = df2[(df2['Sex']!='All') & (df2['Age group']=='All') &
                  (df2['Offence group']=='All') & (df2['Police Force Area']=='All')]

df2_gender_eth = df2_gender.groupby(['Sex','Ethnicity'])['Denominator (Prosecutions at Magistrates Court)'].sum().unstack()

df2_gender_eth.plot(kind='bar', figsize=(12, 6), colormap='Pastel1', edgecolor='black')
plt.title('Prosecutions by Sex and Ethnicity (2009–2017)', pad=15)
plt.xlabel('Sex')
plt.ylabel('Total Prosecutions')
plt.xticks(rotation=0)
plt.gca().yaxis.set_major_formatter(fmt_comma)
plt.legend(title='Ethnicity', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

total_male   = df2[df2['Sex']=='Male']['Denominator (Prosecutions at Magistrates Court)'].sum()
total_female = df2[df2['Sex']=='Female']['Denominator (Prosecutions at Magistrates Court)'].sum()
print(f"Male prosecutions:   {total_male:,}")
print(f"Female prosecutions: {total_female:,}")
print(f"Male-to-female ratio: {total_male/total_female:.1f}x")

**Key observations:**
- Males account for approximately **6.2x** more prosecutions than females across all ethnicities.
- The gender gap is present across all ethnic groups, though the ratio varies.
- **Additional inference:** This is one of the most consistent patterns in criminal justice data globally. The gender gap in prosecution is larger than any ethnic gap, yet it receives comparatively less policy attention. Female prosecution rates are notably higher for Black women relative to other female ethnic groups — a dimension worth further investigation.

### 4.5 Conviction Rate by Offence Group

In [ ]:
df2_offence = df2[(df2['Sex']=='All') & (df2['Age group']=='All') &
                   (df2['Police Force Area']=='All') & (df2['Ethnicity']!='All') &
                   (df2['Offence group']!='All')].copy()

df2_offence_cr = df2_offence.groupby('Offence group').agg(
    Prosecutions=('Denominator (Prosecutions at Magistrates Court)', 'sum'),
    Convictions=('Numerator (Convictions and Magistrates Court and Crown Court)', 'sum')
).reset_index()
df2_offence_cr['conviction_rate'] = df2_offence_cr['Convictions'] / df2_offence_cr['Prosecutions'] * 100
df2_offence_cr = df2_offence_cr.sort_values('conviction_rate', ascending=True)

colors = ['#d9534f' if r < 65 else '#f0ad4e' if r < 80 else '#5cb85c'
          for r in df2_offence_cr['conviction_rate']]

plt.figure(figsize=(12, 7))
bars = plt.barh(df2_offence_cr['Offence group'], df2_offence_cr['conviction_rate'],
                color=colors, edgecolor='black')
for b in bars:
    plt.text(b.get_width() + 0.5, b.get_y() + b.get_height()/2,
             f"{b.get_width():.1f}%", va='center', fontsize=13)
plt.axvline(x=80, color='grey', linestyle='--', alpha=0.7, label='80% threshold')
plt.xlabel('Conviction Rate (%)')
plt.title('Conviction Rate by Offence Group (All Ethnicities, 2009–2017)', pad=15)
plt.xlim(0, 105)
plt.legend()
plt.tight_layout()
plt.show()

**Key observations:**
- **Public order offences** and **drug offences** have the highest conviction rates (~90–92%) — once charged, these almost always result in conviction.
- **Sexual offences** have the lowest conviction rate at ~52%, followed by **violence against the person** at ~62%.
- **Robbery** also sits below 65%, reflecting the evidential difficulty of proving robbery beyond reasonable doubt.
- **Additional inference:** The very high conviction rate for drug offences (90%) is significant when read alongside the stop and search data. Drug searches are common — and when they result in prosecution, they almost always result in conviction. This makes drug stop and searches among the most consequential for individuals stopped.

### 4.6 Prosecutions by Ethnicity and Age Group

In [ ]:
df2_age = df2[(df2['Sex']=='All') & (df2['Age group']!='All') &
               (df2['Offence group']=='All') & (df2['Police Force Area']=='All') &
               (df2['Ethnicity']!='All')]

df2_age_grp = df2_age.groupby(['Age group','Ethnicity']).agg(
    Prosecutions=('Denominator (Prosecutions at Magistrates Court)', 'sum')
).reset_index()

pivot_age = df2_age_grp.pivot(index='Age group', columns='Ethnicity', values='Prosecutions')
age_order = ['Juveniles','Young adults','Adults']
pivot_age = pivot_age.reindex(age_order)

pivot_age.plot(kind='bar', figsize=(12,6), colormap='Pastel1', edgecolor='black')
plt.title('Prosecutions by Age Group and Ethnicity (2009–2017)', pad=15)
plt.xlabel('Age Group')
plt.ylabel('Total Prosecutions')
plt.xticks(rotation=0)
plt.gca().yaxis.set_major_formatter(fmt_comma)
plt.legend(title='Ethnicity', bbox_to_anchor=(1.01,1))
plt.tight_layout()
plt.show()

# Proportion of adult prosecutions for each ethnicity
print("\nAdult share of total prosecutions by ethnicity:")
total_by_eth = df2_age_grp.groupby('Ethnicity')['Prosecutions'].sum()
adult_by_eth = df2_age_grp[df2_age_grp['Age group']=='Adults'].set_index('Ethnicity')['Prosecutions']
print((adult_by_eth / total_by_eth * 100).round(1).sort_values(ascending=False))

**Key observations:**
- Adults account for the overwhelming majority of prosecutions across all ethnicities.
- The juvenile-to-adult escalation is steepest for White and Black groups.
- **Additional inference:** The relatively higher juvenile proportion for Black individuals compared to other groups may reflect differential enforcement patterns at younger ages, or socioeconomic factors that increase contact with the justice system earlier in life — both warrant further investigation with social deprivation data.

## 5. Stop and Search Analysis (df3)

### 5.1 National Stop and Search Trend (2006–2023)

The stop and search dataset extends further back and forward than the prosecutions data, allowing us to observe the full arc of S&S policy in England and Wales.

In [ ]:
df3_nat_all = df3[(df3['geography']=='All - excluding BTP') &
                   (df3['Ethnicity']=='All')].groupby('year')['number_of_stop_and_searches'].sum().reset_index()

plt.figure(figsize=(15, 6))
plt.fill_between(df3_nat_all['year'], df3_nat_all['number_of_stop_and_searches'],
                 alpha=0.3, color='steelblue')
plt.plot(df3_nat_all['year'], df3_nat_all['number_of_stop_and_searches'],
         color='steelblue', marker='o', linewidth=2.5)
plt.gca().yaxis.set_major_formatter(fmt_comma)
plt.axvline(x=2014, color='crimson', linestyle='--', alpha=0.8, label='Reduce S&S policy (2014)')
plt.axvline(x=2018, color='darkorange', linestyle='--', alpha=0.8, label='County Lines crackdown (2018)')
plt.title('Total Stop and Searches — England & Wales (2006–2023)', pad=15)
plt.xlabel('Year')
plt.ylabel('Number of Stop and Searches')
plt.legend()
plt.tight_layout()
plt.show()

print("\nPeak year:", df3_nat_all.loc[df3_nat_all['number_of_stop_and_searches'].idxmax(), 'year'])
print("Trough year:", df3_nat_all.loc[df3_nat_all['number_of_stop_and_searches'].idxmin(), 'year'])
print("\nFull trend:")
print(df3_nat_all.to_string(index=False))

**Key observations:**
- Stop and searches peaked in **2009** at over 4 million nationally, then declined sharply.
- A major drop occurs post-2014 following the government's *Best Use of Stop and Search* scheme designed to reduce disproportionate use.
- A secondary rise begins from 2018, linked to county lines drug enforcement and knife crime initiatives.
- **Additional inference:** The 2014–2017 period saw the most dramatic reduction in S&S in recent history. Crucially, crime rates did not spike during this period — challenging the assumption that high S&S volumes are necessary for public safety, and supporting the argument that much of the pre-2014 S&S activity was not proportionate to genuine intelligence.

### 5.2 Stop and Search Disproportionality — Black vs White Rate per 1,000

In [ ]:
df3_bw = df3[(df3['geography']=='All - excluding BTP') &
              (df3['Ethnicity'].isin(['Black','White']))].groupby(
    ['year','Ethnicity'])['rate_per_1_000_population_by_ethnicity'].mean().unstack().reset_index()
df3_bw['ratio'] = df3_bw['Black'] / df3_bw['White']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

ax1.plot(df3_bw['year'], df3_bw['Black'], color='#d62728', marker='o', linewidth=2, label='Black')
ax1.plot(df3_bw['year'], df3_bw['White'], color='steelblue', marker='s', linewidth=2, label='White')
ax1.set_ylabel('Rate per 1,000 population')
ax1.set_title('Stop and Search Rate per 1,000 by Ethnicity', pad=10)
ax1.legend()
ax1.yaxis.set_major_formatter(fmt_comma)

ax2.bar(df3_bw['year'], df3_bw['ratio'], color='#ff7f0e', edgecolor='black', alpha=0.8)
ax2.axhline(y=1, color='black', linestyle='--')
ax2.set_xlabel('Year')
ax2.set_ylabel('Black:White Ratio')
ax2.set_title('Disproportionality Ratio — Black vs White Stop and Search Rate', pad=10)
for i, row in df3_bw.iterrows():
    ax2.text(row['year'], row['ratio'] + 0.1, f"{row['ratio']:.1f}x",
             ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nBlack:White S&S rate ratio over time:")
print(df3_bw[['year','Black','White','ratio']].round(2).to_string(index=False))

**Key observations:**
- Black individuals are stopped at a rate consistently **4 to 9.5 times higher** than White individuals per 1,000 population.
- The ratio *worsened* after 2014 even as absolute volumes fell — meaning the reduction in S&S disproportionately benefited White individuals.
- The peak disproportionality ratio of **9.6x** occurred in 2017.
- **Additional inference:** This is arguably the most striking finding in the entire dataset. A shrinking pool of total stops becoming *more* concentrated on Black individuals suggests that the S&S reduction was not applied equally across ethnic groups. Even as overall volumes dropped, Black individuals continued to bear a disproportionate share of police contact — raising serious questions about whether reforms addressed the structural causes of disproportionality.

### 5.3 Rate per 1,000 Across All Broad Ethnicities

In [ ]:
def harmonise_eth(val):
    if val in ['Asian','Bangladeshi','Chinese','Arab','Indian','Pakistani','Any Other Asian Background']:
        return 'Asian'
    elif val in ['Black','Black African','Black Caribbean','Any Other Black Background']:
        return 'Black'
    elif val in ['Other - Including Chinese','Other']:
        return 'Other inc Chinese'
    elif val in ['Mixed','Mixed White and Asian','Mixed White and Black African',
                  'Mixed White and Black Caribbean','Any Other Mixed/Multiple Ethnic Background',
                  'Any Other Ethnic Background']:
        return 'Mixed'
    elif val in ['White','White British','White Irish','Roma','Any Other White Background',
                  'Gypsy or Irish Traveller']:
        return 'White'
    else:
        return None

df3_eth = df3.copy()
df3_eth['Ethnicity_broad'] = df3_eth['Ethnicity'].apply(harmonise_eth)
df3_eth = df3_eth[df3_eth['Ethnicity_broad'].notna()]

df3_rate = df3_eth[df3_eth['geography']=='All - excluding BTP'].groupby(
    ['year','Ethnicity_broad'])['rate_per_1_000_population_by_ethnicity'].mean().unstack().reset_index()

plt.figure(figsize=(15, 6))
colors_eth = {'Asian':'#ff7f0e','Black':'#d62728','Mixed':'#9467bd',
              'Other inc Chinese':'#8c564b','White':'steelblue'}
for eth in df3_rate.columns[1:]:
    if eth in colors_eth:
        plt.plot(df3_rate['year'], df3_rate[eth], marker='o', linewidth=2,
                 label=eth, color=colors_eth[eth])
plt.title('Stop & Search Rate per 1,000 Population by Broad Ethnicity (2006–2023)', pad=15)
plt.xlabel('Year')
plt.ylabel('Rate per 1,000')
plt.legend(title='Ethnicity')
plt.tight_layout()
plt.show()

**Key observations:**
- Black individuals have the highest S&S rate per 1,000 across the entire 2006–2023 period.
- Asian rates are the second highest, notably elevated post-2018.
- White rates are the lowest throughout.
- **Additional inference:** The post-2018 rise in Asian S&S rates coincides with county lines and knife crime enforcement, which has been concentrated in urban areas with higher Asian and Black populations. This suggests that enforcement policy choices — not just individual officer decisions — drive disproportionality.

## 6. Cross-Dataset Analysis

### 6.1 Stop and Search vs. Prosecutions by Ethnicity (Overlap Period: 2009–2017)

In [ ]:
# Harmonise df3 ethnicity and aggregate
df3_eth_agg = df3_eth.copy()
df3_eth_agg['Time'] = df3_eth_agg['year']
df3_eth_time = df3_eth_agg.groupby(['Time','Ethnicity_broad'])['number_of_stop_and_searches'].sum().reset_index()
df3_eth_time.columns = ['Time','Ethnicity','SaS']

# df2: all filter
df2_eth_time = df2[(df2['Sex']=='All') & (df2['Age group']=='All') &
                   (df2['Offence group']=='All') & (df2['Police Force Area']=='All') &
                   (df2['Ethnicity']!='All')].copy()
df2_eth_time = df2_eth_time.groupby(['Time','Ethnicity']).agg(
    Prosecutions=('Denominator (Prosecutions at Magistrates Court)', 'sum'),
    Convictions=('Numerator (Convictions and Magistrates Court and Crown Court)', 'sum')
).reset_index()

# Merge on Time + Ethnicity
merged = pd.merge(df3_eth_time, df2_eth_time, on=['Time','Ethnicity'], how='inner')
merged['sas_to_prosecution_ratio'] = merged['SaS'] / merged['Prosecutions']

print("Merged dataset shape:", merged.shape)
print("\nAverage Stop & Search to Prosecution ratio by ethnicity:")
print(merged.groupby('Ethnicity')['sas_to_prosecution_ratio'].mean().sort_values().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: S&S vs Prosecutions scatter
for eth in merged['Ethnicity'].unique():
    sub = merged[merged['Ethnicity']==eth]
    axes[0].scatter(sub['SaS'], sub['Prosecutions'], label=eth, s=80, alpha=0.8)
axes[0].set_xlabel('Number of Stop & Searches')
axes[0].set_ylabel('Number of Prosecutions')
axes[0].set_title('Stop & Search vs Prosecutions by Ethnicity', pad=12)
axes[0].xaxis.set_major_formatter(fmt_comma)
axes[0].yaxis.set_major_formatter(fmt_comma)
axes[0].legend(title='Ethnicity')

# Right: Ratio bar chart
ratio_mean = merged.groupby('Ethnicity')['sas_to_prosecution_ratio'].mean().sort_values(ascending=False)
axes[1].barh(ratio_mean.index, ratio_mean.values, color=pastel, edgecolor='black')
for i, v in enumerate(ratio_mean.values):
    axes[1].text(v + 0.2, i, f'{v:.1f}x', va='center', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Stop & Searches per Prosecution')
axes[1].set_title('Average S&S Required per Prosecution by Ethnicity', pad=12)

plt.tight_layout()
plt.show()

**Key observations:**
- White individuals require the most stop and searches per prosecution — meaning S&S of White individuals is the least efficient at generating prosecutions.
- Black individuals have the lowest ratio — S&S of Black individuals is most likely to result in prosecution.
- **Additional inference:** This finding is paradoxical and important. Despite Black individuals being stopped at up to 9.5x the rate of White individuals, each stop is *more* likely to result in prosecution. One interpretation: S&S of Black individuals is more targeted (intelligence-led), while S&S of White individuals is more speculative. Another interpretation: prosecution thresholds may differ by ethnicity. Both demand scrutiny.

### 6.2 Conviction Rate by Offence Group and Ethnicity (Heatmap)

In [ ]:
df2_heat = df2[(df2['Sex']=='All') & (df2['Age group']=='All') &
               (df2['Police Force Area']=='All') & (df2['Ethnicity']!='All') &
               (df2['Offence group']!='All')].copy()
df2_heat['conviction_rate'] = (df2_heat['Numerator (Convictions and Magistrates Court and Crown Court)'] /
                                df2_heat['Denominator (Prosecutions at Magistrates Court)'] * 100)
pivot_heat = df2_heat.groupby(['Ethnicity','Offence group'])['conviction_rate'].mean().unstack().round(1)

plt.figure(figsize=(14, 6))
sns.heatmap(pivot_heat, annot=True, fmt='.1f', cmap='RdYlGn', linewidths=0.5,
            vmin=50, vmax=95, cbar_kws={'label': 'Conviction Rate (%)'})
plt.title('Conviction Rate (%) by Ethnicity and Offence Group', pad=15)
plt.xlabel('Offence Group')
plt.ylabel('Ethnicity')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

**Key observations:**
- Sexual offences have the lowest conviction rate for all ethnicities, with Asian individuals showing the lowest (~47%).
- Drug offences show relatively uniform, high conviction rates across all ethnicities (~88–92%).
- White individuals have consistently higher conviction rates than other groups across most offence categories.
- **Additional inference:** The heatmap reveals that the ethnicity-conviction rate gap is not uniform across offences. For violent offences and robbery, the gap between White and non-White conviction rates is largest — suggesting that case strength differences, witness cooperation rates, or jury composition effects may operate differently across offence types.

### 6.3 Police Force Area — Prosecution Volume vs. S&S Volume

In [ ]:
# Aggregate df2 by PFA
df2_pfa = df2[(df2['Sex']=='All') & (df2['Age group']=='All') &
              (df2['Offence group']=='All') & (df2['Ethnicity']!='All') &
              (~df2['Police Force Area'].isin(['All']))].groupby('Police Force Area').agg(
    Prosecutions=('Denominator (Prosecutions at Magistrates Court)', 'sum')
).reset_index()

# Clean up geography names to match
df3_pfa = df3[(df3['Ethnicity']=='All') & (~df3['geography'].str.startswith('All'))].groupby(
    'geography')['number_of_stop_and_searches'].sum().reset_index()
df3_pfa.columns = ['Police Force Area','SaS']

# Standardise names for merge
df3_pfa['Police Force Area'] = df3_pfa['Police Force Area'].str.replace(' & ',' and ').str.strip()
df2_pfa['Police Force Area'] = df2_pfa['Police Force Area'].str.strip()

pfa_merged = pd.merge(df2_pfa, df3_pfa, on='Police Force Area', how='inner')
pfa_merged = pfa_merged.sort_values('Prosecutions', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(pfa_merged))
w = 0.4
ax.bar(x - w/2, pfa_merged['Prosecutions'], width=w, label='Prosecutions',
       color='thistle', edgecolor='black')
ax.bar(x + w/2, pfa_merged['SaS'],          width=w, label='Stop & Searches',
       color='salmon',  edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(pfa_merged['Police Force Area'], rotation=45, ha='right', fontsize=11)
ax.yaxis.set_major_formatter(fmt_comma)
ax.set_title('Top 15 Police Force Areas — Prosecutions vs Stop & Searches', pad=15)
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print("\nTop 5 Police Force Areas by Prosecutions:")
print(pfa_merged[['Police Force Area','Prosecutions','SaS']].head(5).to_string(index=False))

**Key observations:**
- The Metropolitan Police Area leads by a large margin in both prosecutions and stop and searches.
- Greater Manchester is second in prosecutions but its S&S volume is proportionally much lower.
- **Additional inference:** Areas with a high S&S-to-prosecution ratio (many stops, fewer prosecutions) may be using S&S as a general deterrence tool rather than as evidence-led policing. Areas with a lower ratio are more efficiently converting stops into criminal justice outcomes.

## 7. Conclusions

### Key Findings

1. **Demographic baseline:** White groups make up 81.7% of England and Wales' population. All findings on prosecution and stop and search volumes must be interpreted against this baseline — absolute counts alone are misleading.

2. **Declining prosecutions, rising conviction rates:** Total prosecutions fell from 2009 to 2017, while conviction rates rose from ~76% to ~83%. This points to more selective, evidence-based charging decisions over time.

3. **Conviction rate gap by ethnicity:** White and Other inc. Chinese individuals have the highest conviction rates. Asian and Mixed individuals have the lowest — a persistent gap that warrants further research into case quality and legal representation.

4. **Gender gap:** Males account for 6.2x more prosecutions than females. The gender gap is larger than any ethnic gap observed in this data.

5. **Offence-level conviction rates:** Public order and drug offences have ~90% conviction rates once charged. Sexual offences and robbery are below 65%, reflecting evidential difficulty.

6. **Stop and search disproportionality:** Black individuals are stopped at 4–9.5x the rate of White individuals per 1,000 population. The disproportionality ratio *worsened* after 2014 even as total volumes fell.

7. **S&S efficiency paradox:** Despite being stopped far more often, Black individuals have the lowest number of S&S required per prosecution — suggesting stops are more targeted or that prosecution thresholds differ. White individuals require the most stops per prosecution.

8. **Metropolitan Police and Greater Manchester** dominate in both prosecution volumes and stop and search activity.

---

## 8. Limitations

- Population data is from 2021 only and cannot serve as a time-varying denominator for the 2006–2023 S&S data.
- Geographic name mismatches between datasets limited the scope of PFA-level merged analysis.
- The prosecutions dataset covers only 2009–2017, preventing direct comparison with the full S&S timeline.
- Many-to-many merges on ethnicity across datasets required careful deduplication.
- All ethnic group findings involving absolute counts are confounded by population size differences.

---

## 9. Recommendations

1. Normalise all prosecution and conviction figures by population size (rate per 100,000) to enable fair cross-ethnic comparison.
2. Extend the prosecutions dataset beyond 2017 to align with the S&S data and capture post-county-lines trends.
3. Include deprivation index and socioeconomic data as control variables before drawing causal conclusions about ethnicity and criminal justice outcomes.
4. Standardise geographic naming conventions across data.gov.uk datasets to facilitate automated merging.
5. Disaggregate by both sex and ethnicity simultaneously in future analyses — the interaction between these two dimensions is currently underexplored.

## 10. Methodology and References

### Methodology Summary
- Datasets sourced from data.gov.uk; imported using `pd.read_csv` with latin-1 encoding.
- Null values handled via column-level dropping (>50% threshold), row deletion (prosecutions denominator), zero-fill (counts/rates), and median imputation (population).
- A derived `conviction_rate` metric was computed as Convictions / Prosecutions × 100.
- Ethnicity labels in df3 were harmonised to the five-category ONS classification to enable cross-dataset merging.
- Financial year labels in df3 (e.g. 2009/10) were converted to integer years using the first year of each period.
- All visualisations produced using `matplotlib` and `seaborn`. Colour palettes chosen for accessibility and clarity.

### References
- [learnpython.org](https://www.learnpython.org/)
- [GeeksforGeeks — Choosing the Right Chart Type](https://www.geeksforgeeks.org/data-visualization/choosing-the-right-chart-type-a-technical-guide/)
- [data.gov.uk — Population by Ethnicity and Region](https://www.data.gov.uk)
- [data.gov.uk — Prosecutions and Convictions](https://www.data.gov.uk)
- [data.gov.uk — Stop and Search 2006–2023](https://www.data.gov.uk)